# PM10 — تحلیل حساسیت پیشین‌ها، نوت‌بوک ۳ (نسخه GPU و بهینه‌شده)
## اجرای هر سه حالت پیشین، ذخیره مستقل و ادامه پس از قطع کگل


In [ ]:
import importlib.util, subprocess, sys

required = {
    'pyro': 'pyro-ppl',
    'openpyxl': 'openpyxl',
}
for module_name, package_name in required.items():
    if importlib.util.find_spec(module_name) is None:
        print(f'Installing {package_name} ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package_name])

In [ ]:
from pathlib import Path

USE_GPU = True
USE_JIT = True
NUMERIC_DTYPE = 'float64'   
GPU_DEVICE_INDEX = 0

PRIOR_INDICES_TO_RUN = [0, 1, 2]
TEST_MODE = True       # اجرای مقاله: False
SKIP_EXISTING = True
COPY_PREVIOUS_RESULTS = True
SEED = 1405

PHI_FIXED_OVERRIDE = None

PRIOR_SPECS = [
    {
        'prior_index': 0,
        'prior_name': 'base',
        'prior_label_fa': 'پیشین پایه',
        'lambda_sd': 5.0,
        'sigma_theta_kind': 'halfnormal',
        'sigma_theta_scale': 5.0,
        'prior_text': 'lambda ~ Normal(0, 5^2); sigma_theta ~ HalfNormal(5)',
    },
    {
        'prior_index': 1,
        'prior_name': 'alternative_1',
        'prior_label_fa': 'پیشین جایگزین اول',
        'lambda_sd': 10.0,
        'sigma_theta_kind': 'gamma',
        'sigma_theta_shape': 2.0,
        'sigma_theta_rate': 0.2,
        'prior_text': 'lambda ~ Normal(0, 10^2); sigma_theta ~ Gamma(shape=2, rate=0.2)',
    },
    {
        'prior_index': 2,
        'prior_name': 'alternative_2',
        'prior_label_fa': 'پیشین جایگزین دوم',
        'lambda_sd': 10.0,
        'sigma_theta_kind': 'gamma',
        'sigma_theta_shape': 1.5,
        'sigma_theta_rate': 0.15,
        'prior_text': 'lambda ~ Normal(0, 10^2); sigma_theta ~ Gamma(shape=1.5, rate=0.15)',
    },
]

if any(i not in range(len(PRIOR_SPECS)) for i in PRIOR_INDICES_TO_RUN):
    raise ValueError('شماره پیشین‌ها باید 0، 1 یا 2 باشد.')

MODE_TAG = 'test' if TEST_MODE else 'final'
SETTINGS = dict(
    warmup=100 if TEST_MODE else 700,
    draws=100 if TEST_MODE else 1500,
    chains=1 if TEST_MODE else 2,
    target_accept=0.95,
    max_tree_depth=10 if TEST_MODE else 12,
)

BASE_OUT = Path('/kaggle/working/PM10_prior_sensitivity_GPU')
MODE_OUT = BASE_OUT / MODE_TAG
MODE_OUT.mkdir(parents=True, exist_ok=True)

print('Prior indices:', PRIOR_INDICES_TO_RUN)
print('Mode:', MODE_TAG)
print('Settings:', SETTINGS)

In [ ]:
import os, glob, json, pickle
from pathlib import Path
import numpy as np
import pandas as pd

PREP_PATH_OVERRIDE = None  
def find_prepared_file():
    if PREP_PATH_OVERRIDE is not None:
        p = Path(PREP_PATH_OVERRIDE)
        if not p.exists():
            raise FileNotFoundError(p)
        return p
    hits = (
        glob.glob('/kaggle/working/**/pm10_prepared.pkl', recursive=True)
        + glob.glob('/kaggle/input/**/pm10_prepared.pkl', recursive=True)
    )
    if not hits:
        raise FileNotFoundError(
            'pm10_prepared.pkl پیدا نشد. خروجی Notebook 1 اصلی را به‌صورت Dataset به این Notebook اضافه کنید.'
        )
    return Path(hits[0])

PREP_PATH = find_prepared_file()
with open(PREP_PATH, 'rb') as f:
    prep = pickle.load(f)

print('Prepared file:', PREP_PATH)
print('Metadata:', prep.get('meta', {}))

In [ ]:
import math, time, gc, random, shutil
from functools import partial
import numpy as np
import pandas as pd
import torch
import pyro
import pyro.distributions as dist
from pyro.infer import MCMC, NUTS
from pyro.infer.mcmc.util import diagnostics as sample_diagnostics
from scipy.stats import norm

DEVICE = torch.device(
    f'cuda:{GPU_DEVICE_INDEX}' if USE_GPU and torch.cuda.is_available() else 'cpu'
)
if NUMERIC_DTYPE == 'float64':
    TORCH_DTYPE = torch.float64
elif NUMERIC_DTYPE == 'float32':
    TORCH_DTYPE = torch.float32
else:
    raise ValueError("NUMERIC_DTYPE must be 'float64' or 'float32'.")

pyro.set_rng_seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.set_default_dtype(TORCH_DTYPE)

print('Torch version:', torch.__version__)
print('Selected device:', DEVICE)
print('Numeric dtype:', TORCH_DTYPE)
print('JIT enabled:', USE_JIT)
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(DEVICE))
    print('CUDA version:', torch.version.cuda)
else:
    print('WARNING: CUDA is unavailable; this notebook will run on CPU.')

def log_ndtr(x):
    if hasattr(torch.special, 'log_ndtr'):
        return torch.special.log_ndtr(x)
    cdf = 0.5 * (1 + torch.erf(x / math.sqrt(2)))
    return torch.log(torch.clamp(cdf, min=1e-300))

class TorchFCSN(dist.TorchDistribution):
    """FCSN distribution using fixed, precomputed correlation factors."""
    arg_constraints = {}
    support = dist.constraints.real
    has_rsample = False

    def __init__(
        self, mu, sigma, L, C12, C12_inv, logdet_C, Lambda,
        validate_args=None
    ):
        self.mu = mu
        self.sigma = sigma
        self.L = L
        self.C12 = C12
        self.C12_inv = C12_inv
        self.logdet_C = logdet_C
        self.Lambda = Lambda
        self.dim = mu.shape[-1]
        super().__init__(torch.Size(), torch.Size([self.dim]), validate_args=validate_args)

        one = torch.ones_like(mu)
        delta = Lambda / torch.sqrt(1 + Lambda**2)
        bd = math.sqrt(2 / math.pi) * delta
        st = sigma / torch.sqrt(1 - bd**2 + 1e-12)

        self.one = one
        self.bd = bd
        self.st = st
        self.mux = mu - (bd * st) * (one @ C12)
        self.D = (Lambda / (st + 1e-12)) * C12_inv
        self.logdet_cov = self.dim * torch.log(st**2 + 1e-12) + logdet_C
        self.c1 = st * Lambda / torch.sqrt(1 + Lambda**2)
        self.c2 = st / torch.sqrt(1 + Lambda**2)

    def sample(self, sample_shape=torch.Size()):
        shape = sample_shape + torch.Size([self.dim])
        u1 = torch.randn(shape, dtype=self.mu.dtype, device=self.mu.device)
        u2 = torch.randn(shape, dtype=self.mu.dtype, device=self.mu.device)
        return (
            self.mux
            + (self.c1 * torch.abs(u2)) @ self.C12
            + (self.c2 * u1) @ self.C12
        )

    def log_prob(self, x):
        if x.ndim == 1:
            x = x.unsqueeze(0)
        diff = x - self.mux
        q = torch.linalg.solve_triangular(
            self.L, (diff / (self.st + 1e-12)).T, upper=False
        ).T
        logg = -0.5 * (
            self.dim * math.log(2 * math.pi)
            + self.logdet_cov
            + (q * q).sum(-1)
        )
        z = (x - self.mu) @ self.D + (self.bd * self.Lambda) * self.one
        return self.dim * math.log(2) + logg + log_ndtr(z).sum(-1)

def median_pairwise_distance(points):
    points = np.asarray(points, dtype=float)
    d = np.sqrt(((points[:, None, :] - points[None, :, :]) ** 2).sum(axis=2))
    upper = d[np.triu_indices_from(d, k=1)]
    upper = upper[upper > 0]
    if len(upper) == 0:
        raise ValueError('فاصله مثبت بین گره‌ها پیدا نشد.')
    return float(np.median(upper))

def matern32_corr(knots, phi_fixed=None, jitter=1e-6):
    knots = np.asarray(knots, dtype=float)
    d = np.sqrt(((knots[:, None, :] - knots[None, :, :]) ** 2).sum(axis=2))
    phi = median_pairwise_distance(knots) if phi_fixed is None else float(phi_fixed)
    r = d / (phi + 1e-12)
    C = (1 + math.sqrt(3) * r) * np.exp(-math.sqrt(3) * r)
    C = C + jitter * np.eye(len(knots))
    return C, phi

def make_tensors(prep, C):
    def as_device_array(value):
        return torch.as_tensor(
            np.asarray(value), dtype=TORCH_DTYPE, device=DEVICE
        )

    y = as_device_array(prep['y'])
    X = as_device_array(prep['X'])
    B = as_device_array(prep['B'])
    G = as_device_array(prep['G'])
    Ct = as_device_array(C)

    
    L = torch.linalg.cholesky(Ct)
    C12 = L.T.contiguous()
    eye = torch.eye(Ct.shape[0], dtype=TORCH_DTYPE, device=DEVICE)
    C12_inv = torch.linalg.solve_triangular(C12, eye, upper=True)
    logdet_C = 2.0 * torch.log(torch.diag(L)).sum()

    return y, X, B, G, L, C12, C12_inv, logdet_C


def sigma_theta_prior(spec, ref):
    kind = spec['sigma_theta_kind']
    if kind == 'halfnormal':
        return dist.HalfNormal(ref.new_tensor(spec['sigma_theta_scale']))
    if kind == 'gamma':
        # Pyro/PyTorch: Gamma(concentration, rate). پارامتر دوم نرخ است.
        return dist.Gamma(
            ref.new_tensor(spec['sigma_theta_shape']),
            ref.new_tensor(spec['sigma_theta_rate']),
        )
    raise ValueError(f'Unknown sigma_theta prior: {kind}')


def model(y, X, B, G, L, C12, C12_inv, logdet_C, prior_spec):
    _, P = X.shape
    K = B.shape[1]
    J = G.shape[1]

    z0 = y.new_tensor(0.0)
    sigma_eps = pyro.sample('sigma_eps', dist.HalfNormal(y.new_tensor(5.0)))
    beta0 = pyro.sample('beta0', dist.Normal(z0, y.new_tensor(10.0)))
    beta_rest = pyro.sample(
        'beta_rest',
        dist.Normal(z0, y.new_tensor(2.0)).expand([P - 1]).to_event(1)
    )
    beta = torch.cat([beta0[None], beta_rest])

    # 
    sigma_theta = pyro.sample('sigma_theta', sigma_theta_prior(prior_spec, y))
    Lambda = pyro.sample(
        'Lambda',
        dist.Normal(z0, y.new_tensor(prior_spec['lambda_sd']))
    )

    zero = y.new_zeros(K)
    theta_parts = []
    for j in range(J):
        theta_j = pyro.sample(
            f'theta_{j}',
            TorchFCSN(
                zero, sigma_theta, L, C12, C12_inv, logdet_C, Lambda
            )
        )
        theta_parts.append(theta_j)

    Theta = torch.stack(theta_parts)
    mu = X @ beta + torch.sum((B @ Theta.T) * G, dim=1)
    pyro.sample('y', dist.Normal(mu, sigma_eps).to_event(1), obs=y)

def flatten_samples(grouped_samples):
    out = {}
    for name, tensor in grouped_samples.items():
        arr = tensor.detach().cpu().numpy()
        out[name] = arr.reshape((-1,) + arr.shape[2:])
    return out

def core_draws_dataframe(grouped_samples):
    beta0 = grouped_samples['beta0'].detach().cpu().numpy()
    beta_rest = grouped_samples['beta_rest'].detach().cpu().numpy()
    sigma_theta = grouped_samples['sigma_theta'].detach().cpu().numpy()
    sigma_eps = grouped_samples['sigma_eps'].detach().cpu().numpy()
    lam = grouped_samples['Lambda'].detach().cpu().numpy()

    n_chains, n_draws = beta0.shape[:2]
    rows = []
    for c in range(n_chains):
        frame = pd.DataFrame({
            'chain': c,
            'draw': np.arange(n_draws),
            'beta0': beta0[c],
            'beta1': beta_rest[c, :, 0],
            'beta2': beta_rest[c, :, 1],
            'beta3': beta_rest[c, :, 2],
            'beta4': beta_rest[c, :, 3],
            'sigma_theta': sigma_theta[c],
            'sigma_eps': sigma_eps[c],
            'Lambda': lam[c],
        })
        rows.append(frame)
    return pd.concat(rows, ignore_index=True)

def scalar_summary(grouped_samples):
    core = {
        'beta0': grouped_samples['beta0'],
        'beta_rest': grouped_samples['beta_rest'],
        'sigma_theta': grouped_samples['sigma_theta'],
        'sigma_eps': grouped_samples['sigma_eps'],
        'Lambda': grouped_samples['Lambda'],
    }
    diag = sample_diagnostics(core, group_by_chain=True)
    n_chains = int(grouped_samples['beta0'].shape[0])

    mapping = [
        ('beta0', 'beta0', None),
        ('beta1', 'beta_rest', 0),
        ('beta2', 'beta_rest', 1),
        ('beta3', 'beta_rest', 2),
        ('beta4', 'beta_rest', 3),
        ('sigma_theta', 'sigma_theta', None),
        ('sigma_eps', 'sigma_eps', None),
        ('Lambda', 'Lambda', None),
    ]

    rows = []
    for parameter, site, component in mapping:
        tensor = grouped_samples[site]
        values = (
            tensor[..., component] if component is not None else tensor
        ).detach().cpu().numpy().reshape(-1)

        ess_obj = diag[site]['n_eff']
        ess = (
            float(ess_obj[component].detach().cpu())
            if component is not None
            else float(ess_obj.detach().cpu())
        )

        if n_chains >= 2:
            rhat_obj = diag[site]['r_hat']
            rhat = (
                float(rhat_obj[component].detach().cpu())
                if component is not None
                else float(rhat_obj.detach().cpu())
            )
        else:
            rhat = np.nan

        rows.append({
            'parameter': parameter,
            'mean': float(np.mean(values)),
            'sd': float(np.std(values, ddof=1)),
            'median': float(np.median(values)),
            'q05': float(np.quantile(values, 0.05)),
            'q95': float(np.quantile(values, 0.95)),
            'ESS': ess,
            'Rhat': rhat,
        })
    return pd.DataFrame(rows)

def posterior_mean_matrix(flat, prep, ids):
    X = np.asarray(prep['X'], dtype=float)
    B = np.asarray(prep['B'], dtype=float)
    G = np.asarray(prep['G'], dtype=float)
    J = int(prep['meta']['J'])

    beta = np.column_stack([
        flat['beta0'][ids],
        flat['beta_rest'][ids],
    ])
    Theta = np.stack(
        [flat[f'theta_{j}'][ids] for j in range(J)],
        axis=1
    )
    trend = np.einsum('np,sp->sn', X, beta, optimize=True)
    latent = np.einsum('nk,sjk,nj->sn', B, Theta, G, optimize=True)
    return trend + latent

def logmeanexp(a, axis=0):
    m = np.max(a, axis=axis, keepdims=True)
    return np.squeeze(m, axis=axis) + np.log(
        np.mean(np.exp(a - m), axis=axis)
    )

def full_metrics(prep, flat, max_eval_draws, metric_seed=1405):
    total = len(flat['sigma_eps'])
    use = min(total, max_eval_draws)
    ids = np.linspace(0, total - 1, use).astype(int)

    MU = posterior_mean_matrix(flat, prep, ids)
    y = np.asarray(prep['y'], dtype=float)
    sig = np.asarray(flat['sigma_eps'])[ids]

    LL = (
        -0.5 * np.log(2 * np.pi * sig[:, None] ** 2)
        -0.5 * ((y[None, :] - MU) / sig[:, None]) ** 2
    )
    lppd = float(logmeanexp(LL, axis=0).sum())
    p_waic = float(np.var(LL, axis=0, ddof=1).sum())
    waic = float(-2 * (lppd - p_waic))

    dev = -2 * LL.sum(axis=1)
    d_bar = float(dev.mean())
    mu_bar = MU.mean(axis=0)
    sig_bar = float(sig.mean())
    d_hat = float(-2 * np.sum(norm.logpdf(y, loc=mu_bar, scale=sig_bar)))
    p_d = float(d_bar - d_hat)
    dic = float(d_bar + p_d)

    pred = MU.mean(axis=0)
    rmse = float(np.sqrt(np.mean((y - pred) ** 2)))
    mae = float(np.mean(np.abs(y - pred)))

    rng = np.random.default_rng(metric_seed)
    YREP = MU + rng.normal(size=MU.shape) * sig[:, None]
    lo, hi = np.quantile(YREP, [0.05, 0.95], axis=0)
    coverage = float(np.mean((y >= lo) & (y <= hi)))

    return {
        'lppd': lppd,
        'WAIC': waic,
        'p_WAIC': p_waic,
        'DIC': dic,
        'p_D': p_d,
        'RMSE_in': rmse,
        'MAE_in': mae,
        'coverage90_in': coverage,
        'eval_draws': int(use),
    }

def sampler_diagnostics_one(mcmc, kernel):
    diag_all = mcmc.diagnostics()

    div_obj = diag_all.get('divergences', {})
    if isinstance(div_obj, dict):
        n_divergences = int(sum(len(v) for v in div_obj.values()))
    else:
        n_divergences = int(len(div_obj)) if div_obj is not None else 0

    acc_obj = diag_all.get('acceptance rate', {})
    if isinstance(acc_obj, dict):
        acc_values = [float(v) for v in acc_obj.values()]
    elif acc_obj is None:
        acc_values = []
    else:
        acc_values = [float(acc_obj)]

    try:
        step_size = float(kernel.step_size)
    except Exception:
        step_size = np.nan

    return n_divergences, acc_values, step_size


def make_kernel(prior_spec):
    return NUTS(
        partial(model, prior_spec=prior_spec),
        target_accept_prob=SETTINGS['target_accept'],
        max_tree_depth=SETTINGS['max_tree_depth'],
        jit_compile=USE_JIT,
        ignore_jit_warnings=True,
    )


def run_prior(prep, prior_spec, settings, phi_override, seed):
    C, phi_fixed = matern32_corr(prep['knots'], phi_override)
    args = make_tensors(prep, C)
    y = args[0]

    print('Data device:', y.device)
    print('Data dtype:', y.dtype)
    if DEVICE.type == 'cuda':
        print('GPU allocated before MCMC (GB):', torch.cuda.memory_allocated(DEVICE) / 1e9)

    pyro.clear_param_store()
    np.random.seed(seed)

    start = time.time()
    grouped_parts = []
    divergence_counts = []
    acceptance_values = []
    step_sizes = []

    
    if DEVICE.type == 'cuda' and settings['chains'] > 1:
        for chain_id in range(settings['chains']):
            chain_seed = seed + 1000 * chain_id
            pyro.clear_param_store()
            pyro.set_rng_seed(chain_seed)
            torch.manual_seed(chain_seed)
            torch.cuda.manual_seed_all(chain_seed)

            kernel = make_kernel(prior_spec)
            mcmc = MCMC(
                kernel,
                warmup_steps=settings['warmup'],
                num_samples=settings['draws'],
                num_chains=1,
                disable_progbar=False,
            )
            print(f'GPU sequential chain {chain_id + 1}/{settings["chains"]}')
            mcmc.run(*args)

            one = {
                k: v.detach().cpu()
                for k, v in mcmc.get_samples(group_by_chain=True).items()
            }
            grouped_parts.append(one)
            div, acc, step = sampler_diagnostics_one(mcmc, kernel)
            divergence_counts.append(div)
            acceptance_values.extend(acc)
            step_sizes.append(step)

            del mcmc, kernel, one
            torch.cuda.empty_cache()

        grouped = {
            key: torch.cat([part[key] for part in grouped_parts], dim=0)
            for key in grouped_parts[0]
        }
    else:
        pyro.set_rng_seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

        kernel = make_kernel(prior_spec)
        mcmc_kwargs = dict(
            warmup_steps=settings['warmup'],
            num_samples=settings['draws'],
            num_chains=settings['chains'],
            disable_progbar=False,
        )
        if settings['chains'] > 1:
            mcmc_kwargs['mp_context'] = 'fork'

        mcmc = MCMC(kernel, **mcmc_kwargs)
        mcmc.run(*args)
        grouped = {
            k: v.detach().cpu()
            for k, v in mcmc.get_samples(group_by_chain=True).items()
        }
        div, acc, step = sampler_diagnostics_one(mcmc, kernel)
        divergence_counts.append(div)
        acceptance_values.extend(acc)
        step_sizes.append(step)

    runtime = time.time() - start

    if DEVICE.type == 'cuda':
        peak_memory_gb = torch.cuda.max_memory_allocated(DEVICE) / 1e9
    else:
        peak_memory_gb = np.nan

    return grouped, {
        'phi_fixed': float(phi_fixed),
        'runtime_sec': float(runtime),
        'device': str(DEVICE),
        'gpu_name': torch.cuda.get_device_name(DEVICE) if DEVICE.type == 'cuda' else 'CPU',
        'numeric_dtype': str(TORCH_DTYPE),
        'jit_compile': bool(USE_JIT),
        'gpu_peak_memory_gb': float(peak_memory_gb),
        'n_divergences': int(sum(divergence_counts)),
        'acceptance_rate_mean': float(np.mean(acceptance_values)) if acceptance_values else np.nan,
        'acceptance_rate_min': float(np.min(acceptance_values)) if acceptance_values else np.nan,
        'acceptance_rate_max': float(np.max(acceptance_values)) if acceptance_values else np.nan,
        'final_step_size': float(np.nanmean(step_sizes)) if step_sizes else np.nan,
        'step_size_note': 'mean final step size across sequential chains',
    }

def save_run(run_dir, prep, prior_spec, settings, grouped, run_meta, test_mode):
    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)

    summary = scalar_summary(grouped)
    summary.insert(0, 'prior_name', prior_spec['prior_name'])
    summary.insert(0, 'prior_index', prior_spec['prior_index'])

    flat = flatten_samples(grouped)
    metrics = full_metrics(
        prep,
        flat,
        max_eval_draws=100 if test_mode else 400,
        metric_seed=SEED,
    )
    metric_row = {
        'prior_index': prior_spec['prior_index'],
        'prior_name': prior_spec['prior_name'],
        'prior_text': prior_spec['prior_text'],
        'test_mode': bool(test_mode),
        **settings,
        **run_meta,
        **metrics,
    }
    metrics_df = pd.DataFrame([metric_row])

    core_draws = core_draws_dataframe(grouped)
    core_draws.insert(0, 'prior_name', prior_spec['prior_name'])
    core_draws.insert(0, 'prior_index', prior_spec['prior_index'])

    summary.to_csv(run_dir / 'posterior_summary.csv', index=False)
    metrics_df.to_csv(run_dir / 'run_metrics.csv', index=False)
    core_draws.to_csv(run_dir / 'core_posterior_draws.csv', index=False)

    with open(run_dir / 'posterior_samples.pkl', 'wb') as f:
        pickle.dump(
            {
                'samples': {k: v.detach().cpu() for k, v in grouped.items()},
                'prior_spec': prior_spec,
                'settings': settings,
                'run_meta': run_meta,
            },
            f,
        )

    config = {
        'prior_spec': prior_spec,
        'settings': settings,
        'test_mode': bool(test_mode),
        'seed': SEED,
        'prepared_file': str(PREP_PATH),
        'prepared_meta': prep.get('meta', {}),
        'run_meta': run_meta,
    }
    with open(run_dir / 'run_config.json', 'w', encoding='utf-8') as f:
        json.dump(config, f, ensure_ascii=False, indent=2, default=str)

    with open(run_dir / 'COMPLETED.json', 'w', encoding='utf-8') as f:
        json.dump(
            {
                'completed': True,
                'prior_index': prior_spec['prior_index'],
                'test_mode': bool(test_mode),
                'draws': settings['draws'],
                'chains': settings['chains'],
                'device_type': DEVICE.type,
                'numeric_dtype': NUMERIC_DTYPE,
                'jit_compile': bool(USE_JIT),
            },
            f,
            indent=2,
        )

    return summary, metrics_df, core_draws

In [ ]:
import glob, shutil

if COPY_PREVIOUS_RESULTS:
    candidates = glob.glob('/kaggle/input/**/prior_*', recursive=True)
    copied = 0
    for src_text in candidates:
        src = Path(src_text)
        if not src.is_dir():
            continue
       
        if MODE_TAG not in [p.name for p in src.parents]:
            continue
        dst = MODE_OUT / src.name
        if not dst.exists():
            shutil.copytree(src, dst)
            copied += 1
    print('Previous run directories copied:', copied)

In [ ]:
all_summary = []
all_metrics = []
failures = []

for prior_index in PRIOR_INDICES_TO_RUN:
    prior_spec = PRIOR_SPECS[prior_index]
    run_dir = MODE_OUT / f"prior_{prior_index}_{prior_spec['prior_name']}"
    completed_file = run_dir / 'COMPLETED.json'

    if SKIP_EXISTING and completed_file.exists():
        with open(completed_file, 'r') as f:
            completed = json.load(f)
        compatible = (
            bool(completed.get('test_mode')) == bool(TEST_MODE)
            and int(completed.get('draws', -1)) == SETTINGS['draws']
            and int(completed.get('chains', -1)) == SETTINGS['chains']
            and completed.get('device_type') == DEVICE.type
            and completed.get('numeric_dtype') == NUMERIC_DTYPE
            and bool(completed.get('jit_compile')) == bool(USE_JIT)
        )
        if compatible:
            print(f'Skipped completed run: {run_dir}')
            all_summary.append(pd.read_csv(run_dir / 'posterior_summary.csv'))
            all_metrics.append(pd.read_csv(run_dir / 'run_metrics.csv'))
            continue

    run_dir.mkdir(parents=True, exist_ok=True)
    with open(run_dir / 'STARTED.json', 'w') as f:
        json.dump(
            {
                'started': True,
                'prior_index': prior_index,
                'test_mode': TEST_MODE,
                'settings': SETTINGS,
            },
            f,
            indent=2,
        )

    print('\n' + '=' * 100)
    print(f"PRIOR {prior_index}: {prior_spec['prior_label_fa']}")
    print(prior_spec['prior_text'])
    print('=' * 100)

    try:
        grouped, run_meta = run_prior(
            prep=prep,
            prior_spec=prior_spec,
            settings=SETTINGS,
            phi_override=PHI_FIXED_OVERRIDE,
            seed=SEED,  
        )
        summary_df, metrics_df, _ = save_run(
            run_dir=run_dir,
            prep=prep,
            prior_spec=prior_spec,
            settings=SETTINGS,
            grouped=grouped,
            run_meta=run_meta,
            test_mode=TEST_MODE,
        )
        all_summary.append(summary_df)
        all_metrics.append(metrics_df)

        del grouped
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    except Exception as exc:
        failure = {'prior_index': prior_index, 'error': repr(exc)}
        failures.append(failure)
        with open(run_dir / 'FAILED.json', 'w') as f:
            json.dump(failure, f, indent=2)
        print('ERROR:', repr(exc))

# 
summary_files = sorted(MODE_OUT.glob('prior_*/posterior_summary.csv'))
metrics_files = sorted(MODE_OUT.glob('prior_*/run_metrics.csv'))

if summary_files:
    combined_summary = pd.concat(
        [pd.read_csv(p) for p in summary_files], ignore_index=True
    ).sort_values(['prior_index', 'parameter'])
    combined_summary.to_csv(MODE_OUT / 'all_posterior_summaries.csv', index=False)
    display(combined_summary)

if metrics_files:
    combined_metrics = pd.concat(
        [pd.read_csv(p) for p in metrics_files], ignore_index=True
    ).sort_values('prior_index')
    combined_metrics.to_csv(MODE_OUT / 'all_run_metrics.csv', index=False)
    display(combined_metrics)

if failures:
    pd.DataFrame(failures).to_csv(MODE_OUT / 'failures.csv', index=False)
    print('Failed runs:', failures)
else:
    print('All requested priors completed successfully.')

print('Outputs:', MODE_OUT)